# Single Image Processing from a LIF Container

This notebook processes 3D plant root images from LIF containers with the following steps:

1. Predicts 3D nuclei labels using CellposeSAM, including anisotropy correction and volume filtering.
2. Generates a rough 3D root mask by combining PanSeg UNet3D boundary predictions with the nuclei labels.
3. Refines the root mask using morphological operations and Euclidean Distance Transform (EDT) guided by nuclei positions.
4. Calculates a nuclei-to-root-surface depth map with anisotropy correction (using voxel spacing from metadata).
5. Assigns root part and tissue layer to each nucleus using KMeans clustering based on geometric and intensity features.

All processing steps are visualized in Napari and results are saved to appropriate output folders.

Image Outputs are written to `RAW_DATA_DIRECTORY/<lif_container_id>/`.

### Environment setup
This cell sets up imports and makes sure the src folder is available in Python path.

Input needed from researcher: run this as-is unless your project folder layout is different.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    get_voxel_spacing_zyx_um,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    generate_rough_root_3d,
    fill_root_3d,
    smooth_outer_root_surface_3d,
    wrap_outer_root_surface
)
from utils.inference import predict_tiled_unet

from utils.feature_extraction import (
    calculate_distance_to_root_surface,
    extract_nuclei_features_per_marker,
    compute_fret_ratios,
    classify_root_cap_nuclei,
    extract_nuclei_depth,
    map_root_body_depth_clusters_to_tissue_layers,
    merge_root_cap_into_tissue_layers
)

from utils.data_viz import (
    map_df_column_to_labels,
    plot_prop_to_3d_centroids
)

import tifffile
import napari
import torch

### Configure experiment inputs
Set core paths and model choices used in the workflow.

Input needed from researcher: update RAW_DATA_DIRECTORY, MODEL_DIR, MIN_MAX_NUCLEI_VOLUME, channel indices, marker definitions, and index for image and container from your experiment.

In [ ]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor, DD)
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor, DA – FRET signal)
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor, AA) – used for nuclei segmentation (and optionally for correction)
MARKERS = (("edCerulean_CTRL", 0, "DD"), ("edCitrine_FRET", 1, "DA"), ("edCitrine_CTRL", 2, "AA"), ("brightfield", 3, "root_structure"))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 2

### Discover LIF containers
This cell scans the raw data directory and lists available .lif containers.

Input needed from researcher: confirm the directory contains the expected files.

In [ ]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

if not lif_containers:
    raise FileNotFoundError(
        f"No .lif containers found in '{RAW_DATA_DIRECTORY}'. "
        "Check RAW_DATA_DIRECTORY and file extension."
    )

if not 0 <= LIF_CONTAINER_INDEX < len(lif_containers):
    raise IndexError(
        f"LIF_CONTAINER_INDEX={LIF_CONTAINER_INDEX} is out of range. "
        f"Valid range is 0..{len(lif_containers)-1}."
    )

lif_containers

### Select one container and image
Choose the container and image index to analyze, then load metadata and image data.

Input needed from researcher: set LIF_CONTAINER_INDEX and LIF_IMAGE_INDEX to the sample you want to inspect in the cells above.

In [ ]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

### Display input channels in Napari
This opens a 3D Napari view of all channels to verify image quality and channel order.

Input needed from researcher: visually confirm that channels and marker names match expectations.

In [ ]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

### Prepare PanSeg input from brightfield
This creates a simulated fluorescence image from brightfield and adds it to the viewer.

Input needed from researcher: check whether the simulated boundaries look suitable for downstream segmentation. 
Otherwise consider using a different model under ./plantseg_models, then update MODEL_DIR accordingly

In [ ]:
# Simulate fluorescently labelled cell walls from brightfield input image
sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


### Nuclei label prediction using CellposeSAM
This step loads existing nuclei labels if present, or predicts and saves new labels.

Input needed from researcher: verify CellposeSAM settings (for example anisotropy/rescale and channel choice).

In [ ]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, lif_image_name, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)



### Extract nuclei features
Build image descriptors and compute per-nucleus morphological and intensity features.

Input needed from researcher: confirm marker/channel mapping reflects your biological labels.

In [ ]:
# Create a dictionary containing all image descriptors
descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
            }

# Extract morphological and intensity features per marker
props_df = extract_nuclei_features_per_marker(nuclei_labels, lif_image, MARKERS, descriptor_dict)

# Calculate FRET ratios and add to props_df
props_df = compute_fret_ratios(props_df, MARKERS)

### Root mask prediction using PanSeg
This step loads a precomputed 3D root mask or predicts one and stores it.

Input needed from researcher: verify model selection and output quality of the root mask.

In [ ]:
# Ensure output directory for this container's 3D root mask
root_mask_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="root_mask")
print(f"3D Root Mask directory: {root_mask_dir}")

# Load precomputed root mask when available; otherwise generate and store it
root_body_3d = load_precomputed_results_if_available(root_mask_dir, lif_image_name, results_type="root_mask")

if root_body_3d is not None:
    print(f"3D root mask already calculated for: {lif_image_name} ...loading")
    # Classify nuclei as belonging to the root cap or the rest of the root structure using clustering.
    props_df = classify_root_cap_nuclei(props_df)
    # Add the precomputed mask to the napari viewer
    viewer.add_image(root_body_3d, name="smooth_root_3d", colormap="green", blending="additive", opacity=0.5)

else:
    # Predict root cell boundary probability maps using a pre-trained UNet3D model
    root_pmaps = predict_tiled_unet(
        raw=sim_fluo_cell_walls,
        input_layout="ZYX",
        model_dir=MODEL_DIR,
        patch=(80, 160, 160),
        patch_halo=(8, 16, 16),
        stride_ratio=0.75,
        batch_size=1,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_amp=True,
    )

    # root_pmaps: (C_out, Z, Y, X)
    viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

    # Generate a rough 3D root mask
    filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)
    # Fill internal gaps inside rough 3D root mask
    filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, slice_aware_filling=True, visualize=True)
    # Smooth root outer surface to remove small protrusions
    smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)
    # Classify nuclei as belonging to the root cap or the rest of the root structure using clustering.
    props_df = classify_root_cap_nuclei(props_df)
    # Wrap the outer root surface so it fits better around the outtermost top nuclei layer (ignore root cap nuclei)
    root_body_3d = wrap_outer_root_surface(nuclei_labels, smooth_root_3d, props_df, percentage_threshold = 5.0, edt_threshold = 15.0, visualize = True)
    # Create path for root mask (used only when saving a newly computed prediction)
    root_mask_path = root_mask_dir / f"{lif_image_name}_root_mask.tif"
    # Save the processed mask
    tifffile.imwrite(root_mask_path, root_body_3d)

### Nuclei depth map generation
This step loads or computes a nuclei depth map and reports flooding metadata if relevant.

Input needed from researcher: review depth map quality and any flooding flags.

In [ ]:
# Ensure output directory for this container's nuclei depth_map
depth_map_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="depth_map")
print(f"Nuclei depth map directory: {depth_map_dir}")

# Load precomputed depth map when available; otherwise generate and store it
nuclei_depth_map = load_precomputed_results_if_available(depth_map_dir, lif_image_name, results_type="depth_map")
is_flooded = False
flooded_planes = []

if nuclei_depth_map is not None:
    print(f"Nuclei depth map already calculated for: {lif_image_name} ...loading")
    # Add the precomputed map to Napari
    viewer.add_image(nuclei_depth_map, name="nuclei_depth_normalized", colormap="viridis", blending="additive")

else:
    # Always compute depth distances with anisotropy correction using metadata voxel spacing (z, y, x) in um.
    # This affects distance computation for depth assignment, not visualization.
    spacing_zyx_um = get_voxel_spacing_zyx_um(xml_metadata)
    print(f"Using spacing-aware depth distance mode with spacing (z,y,x) um = {spacing_zyx_um}.")

    # Calculate distance from each nuclei centroid to the root surface.
    # This will be used to approximate to which tissue layer each nucleus belongs.
    nuclei_depth_map, is_flooded, flooded_planes = calculate_distance_to_root_surface(
        nuclei_labels,
        root_body_3d,
        pad_full_root=False,
        spacing_zyx_um=spacing_zyx_um,
        visualize=True,
    )

    # Create path for nuclei depth map (used only when saving a newly computed prediction)
    nuclei_depth_path = depth_map_dir / f"{lif_image_name}_depth_map.tif"
    # Save the calculated depth map
    tifffile.imwrite(nuclei_depth_path, nuclei_depth_map)

print(f"Flood fill applied: {is_flooded}; flooded planes: {flooded_planes}")

### Depth and tissue layer annotation
Merge depth values into the feature table and map depth clusters to tissue layers.

Input needed from researcher: review cluster-to-layer mapping assumptions for your sample type.

In [ ]:
# Extract depth information from nuclei labels and merge with props_df
depth_df = extract_nuclei_depth(nuclei_labels, nuclei_depth_map)
props_df = props_df.merge(depth_df, on="label")

# Map root body depth clusters to tissue layers
depth_clusters_df = map_root_body_depth_clusters_to_tissue_layers(props_df)

# Merge root cap into tissue layers
props_df = merge_root_cap_into_tissue_layers(props_df, depth_clusters_df)

props_df

### Root tip assignment and distance to tip calculation
Find out longest axis and determine axis extreme closest to the outermost root tip cell.

In [ ]:
import numpy as np

# Unpack the shape to retrieve dimensions
_, y_dim, x_dim = nuclei_labels.shape

# Determine the longest in-plane axis (1 -> y, 2 -> x)
# If dimensions are equal, uses x by default
axis = 2 if x_dim >= y_dim else 1

# Find the corresponding centroid coordinate array:
if axis == 2:
    # X is the longest; use centroid-2
    axis_centroids = props_df["centroid-2"].values
    axis_label = "centroid-2 (X)"
else:
    # Y is the longest; use centroid-1
    axis_centroids = props_df["centroid-1"].values
    axis_label = "centroid-1 (Y)"

# Estimate density across the axis using a histogram
n_bins = min(100, max(10, int(np.sqrt(len(axis_centroids)))))
hist, bin_edges = np.histogram(axis_centroids, bins=n_bins)

# Identify which extreme (min or max) has the higher density near the edge
# Take the first and last 10% of bins as "extremes"
n_extreme = max(1, int(0.1 * n_bins))
min_extreme_density = np.sum(hist[:n_extreme])
max_extreme_density = np.sum(hist[-n_extreme:])

if min_extreme_density >= max_extreme_density:
    extreme_value = bin_edges[0]  # Minimum side has higher density
    extreme_side = "min"
else:
    extreme_value = bin_edges[-1]  # Maximum side has higher density
    extreme_side = "max"

print(f"Longest axis: {axis}. Using {axis_label} to define root tip orientation.")
print(f"Highest centroid density at the {extreme_side} extreme (value: {extreme_value}).")

# --- Begin new snippet: find label closest to extreme and highest centroid-0 ---

# Work with the centroid values as a dataframe for easier selections
import numpy as np

# Find distance from each centroid along axis to the relevant extreme value (min or max)
if extreme_side == "min":
    axis_distances = np.abs(axis_centroids - bin_edges[0])
else:
    axis_distances = np.abs(axis_centroids - bin_edges[-1])

# Find the minimal distance (closest to extreme)
min_distance = np.min(axis_distances)
closest_indices = np.where(axis_distances == min_distance)[0]

# If multiple, pick the one with the highest centroid-0 (Z)
if len(closest_indices) > 1:
    centroid0_values = props_df.iloc[closest_indices]["centroid-0"].values
    highest_centroid0_idx = closest_indices[np.argmax(centroid0_values)]
else:
    highest_centroid0_idx = closest_indices[0]

closest_label = props_df.iloc[highest_centroid0_idx]["label"]
closest_centroid_coords = props_df.iloc[highest_centroid0_idx][["centroid-0", "centroid-1", "centroid-2"]].values

print(f"Label closest to {extreme_side} extreme (and with highest centroid-0 among ties): {closest_label}")
print(f"Centroid coordinates (Z, Y, X): {closest_centroid_coords}")

In [ ]:
# Add a "tip_cell" column that is 1 for the row with 'closest_label', else 0
props_df["tip_cell"] = (props_df["label"] == closest_label).astype(int)

# Calculate Euclidean distance from each centroid to the tip_cell centroid (in 3D)
tip_centroid = closest_centroid_coords.astype(float)

# Extract all centroids as a numpy array for vectorized computation
centroids = props_df[["centroid-0", "centroid-1", "centroid-2"]].values.astype(float)
distances = np.linalg.norm(centroids - tip_centroid, axis=1)

# Normalize distances to [0, 1], avoid division by zero if all distances are the same
dist_min = distances.min()
dist_max = distances.max()
if dist_max - dist_min == 0:
    norm_distances = np.zeros_like(distances)
else:
    norm_distances = (distances - dist_min) / (dist_max - dist_min)

props_df["distance_to_tip"] = norm_distances

### Visualize root cap assignment
Render root cap cluster labels back onto nuclei for quality control.

Input needed from researcher: inspect whether root cap assignment matches anatomy.

In [ ]:
# Visualize root cap nuclei
tip_cluster_id_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="tip_cluster_id",
    colormap="turbo",
    visualize=True
)

### Visualize depth clusters
Render depth cluster IDs on nuclei labels for quick spatial QC.

Input needed from researcher: check for over-segmentation or cluster mixing across layers.

In [ ]:
# Visualize depth clusters
depth_cluster_id_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="depth_cluster_id",
    colormap="turbo",
    visualize=True
)

### Visualize FRET ratios
Render normalized FRET ratios per nucleus for exploratory interpretation.

Input needed from researcher: verify normalization and inspect outliers before analysis.

In [ ]:
# Visualize FRET ratios
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="FRET_ratio_sum_norm_per_image",
    colormap="inferno",
    visualize=True
)

### Visualize root tip assignment and tip cell classification
Render distance to root tip cell per nucleus.

Input needed from researcher: verify correct root tip assignment.

In [ ]:
# Visualize tip cell
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="tip_cell",
    colormap="viridis",
    visualize=True
)

# Visualize distance to tip
fret_img = map_df_column_to_labels(
    nuclei_labels,
    props_df,
    value_column="distance_to_tip",
    colormap="inferno",
    visualize=True
)

### Matplotlib visualization

Besides Napari you can use Matplotlib for visualizing your results in a static graph

In [ ]:
plot_prop_to_3d_centroids(
    props_df,
    value_column="depth_cluster_id",
    colormap="turbo",
    nuclei_labels_shape=nuclei_labels.shape,
    fig_title=None,
    ax_labels=None,
    colormap_vmin=None,
    colormap_vmax=None,
    save_fig=False,
    fig_filename=None,
    fig_savepath=None,
    visualize=True
)

### TODO: workflow tasks

In [ ]:
#TODO: Add pixi tasks for batch processing mode, napari and jupyterlab.

In [ ]:
#TODO: Allow researcher to convert depth_cluster_id_img to labels for editing, and save the results as .tif. Include the same logic as for nuclei_labels, depth_map (but this time for depth_cluster_id)
# if results are pre-computed these are loaded from disk. In this way the manual changes applied by the researcher can influence props_df (need to write this logic, not implemented atm). The logic should overwrite the depth_cluster_id field in props_df according to the label identifier contained in depth_cluster_id_img.

In [ ]:
#TODO: Extract objective used (10x or 20x) from metadata and adjust arguments for minimum and maximum nuclei volume accordingly. Same with morphological operations arguments. 

In [ ]:
#TODO: Reimplement the newly added logic into batch processing mode